In [2]:
using JuMP
import Gurobi

"""
    my_GRBBinvRowi(model::Gurobi.Optimizer, i::Int)

Computes a single tableau row.

`i` is the zero-indexed index of affine constraint row.

## Documentation

See the Gurobi documentation:

https://docs.gurobi.com/projects/optimizer/en/current/reference/c/simplex.html#c.GRBBinvRowi
"""
function my_GRBBinvRowi(model::Gurobi.Optimizer, i::Int)
    pNumVars, pNumConstrs = Ref{Cint}(0), Ref{Cint}(0)
    @assert Gurobi.GRBgetintattr(model, "NumConstrs", pNumConstrs) == 0
    @assert Gurobi.GRBgetintattr(model, "NumVars", pNumVars) == 0
    len = pNumVars[] + pNumConstrs[]
    ind, val = zeros(Cint, len), zeros(Cdouble, len)
    x = Gurobi.GRBsvec(0, pointer(ind), pointer(val))
    pX = Ref(x)
    @assert Gurobi.GRBBinvRowi(model, i, pX) == 0
    modified_x = pX[]
    return ind[1:modified_x.len], val[1:modified_x.len]
end

function my_GRBFSolve(model::Gurobi.Optimizer, b_ind::Vector{Cint}, b_val::Vector{Cdouble})
    # Get number of constraints
    pNumConstrs = Ref{Cint}(0)
    @assert Gurobi.GRBgetintattr(model, "NumConstrs", pNumConstrs) == 0
    m = pNumConstrs[]
    
    # Right-hand side vector (sparse format)
    b_svec = Gurobi.GRBsvec(length(b_ind), pointer(b_ind), pointer(b_val))
    pB = Ref(b_svec)
    
    # Allocate space for result (sparse)
    x_ind = zeros(Cint, m)
    x_val = zeros(Cdouble, m)
    x_svec = Gurobi.GRBsvec(0, pointer(x_ind), pointer(x_val))
    pX = Ref(x_svec)

    # Solve Bx = b
    @assert Gurobi.GRBFSolve(model, pB, pX) == 0

    # Extract sparse solution
    modified_x = pX[]
    return x_ind[1:modified_x.len], x_val[1:modified_x.len]
end

function my_getBasisHead(model::Gurobi.Optimizer)
    # 获取变量数量和约束数量
    pNumVars = Ref{Cint}(0)
    pNumConstrs = Ref{Cint}(0)
    @assert Gurobi.GRBgetintattr(model, "NumVars", pNumVars) == 0
    @assert Gurobi.GRBgetintattr(model, "NumConstrs", pNumConstrs) == 0
    cols, rows = pNumVars[], pNumConstrs[]

    # 分配 bhead 数组，长度为约束数（即 B 的列数）
    bhead = zeros(Cint, rows)

    # 调用 Gurobi API
    @assert Gurobi.GRBgetBasisHead(model, pointer(bhead)) == 0

    # 返回结果
    return bhead, cols
end

my_getBasisHead (generic function with 1 method)

In [3]:
function coefficient_GMI_extended_variable( 
    b_i, 
    val,
    IntSet
)   
    f_j = Dict()
    for (j, a_ij) in val
        f_j[j] = a_ij - floor(a_ij)
    end

    f_0 = b_i - floor(b_i)

    coefficient_set = Dict()
    for j in keys(val) 
        if j in IntSet 
            if f_0 ≥ f_j[j] 
                coefficient_set[j] = f_j[j]/f_0
            else
                coefficient_set[j] = (1 - f_j[j])/(1 - f_0)
            end
        else
            if val[j] ≥ 0 
                coefficient_set[j] = val[j]/f_0
            else
                coefficient_set[j] = - val[j]/(1 - f_0)
            end
        end
    end
    return coefficient_set
end

coefficient_GMI_extended_variable (generic function with 1 method)

In [4]:
function coefficient_GMI_original_variable(
    coefficient_set, 
    cols,
    A,
    b
)   
    rhs = 1
    coef = Dict(k => 0. for k in 1:cols)

    for (j, c_ij) in coefficient_set
        if j ≥ cols
            i = j - cols + 1
            rhs = rhs - c_ij * b[i]
            for k in 1:cols 
                coef[k] = coef[k] - c_ij *  A[i, k]
            end
        end
    end
    return coef, rhs
end

coefficient_GMI_original_variable (generic function with 1 method)

In [5]:
c = [5.5, 2.1]
A = [-1 1; 8 2]
b = [2, 17]
model = direct_model(Gurobi.Optimizer())
@variable(model, x[1:2] >= 0)
@objective(model, Max, c' * x)
@constraint(model, A * x .<= b)
optimize!(model)
grb = backend(model)
b_ind = Cint[0, 1]                 # indices of nonzeros
b_val = Cdouble[2, 17]              # corresponding values

Gurobi Optimizer version 10.0.1 build v10.0.1rc0 (mac64[arm])

CPU model: Apple M3 Max
Thread count: 16 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 2 rows, 2 columns and 4 nonzeros
Model fingerprint: 0xe2091aba
Coefficient statistics:
  Matrix range     [1e+00, 8e+00]
  Objective range  [2e+00, 6e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e+00, 2e+01]
Presolve removed 2 rows and 2 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    1.4080000e+01   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.00 seconds (0.00 work units)
Optimal objective  1.408000000e+01

User-callback calls 48, time in user-callback 0.00 sec


2-element Vector{Float64}:
  2.0
 17.0

In [6]:
x_ind, x_val = my_GRBFSolve(grb, b_ind, b_val)
x_val = Dict(zip(x_ind, x_val))

ind, val = my_GRBBinvRowi(grb, 0)
val = Dict(zip(ind, val))

bhead, cols = my_getBasisHead(grb)

(Int32[1, 0], 2)

In [7]:
x_val

Dict{Int32, Float64} with 2 entries:
  0 => 3.3
  1 => 1.3

In [8]:
# let i be the one with fractional x_val which in an integer
i = 0
basic_index = bhead[i+1]
b_i = x_val[i]

3.3

In [9]:
coefficient_set = coefficient_GMI_extended_variable( 
    b_i,
    val,
    [0,1,2,3]##TODO: integer set
)

Dict{Any, Any} with 3 entries:
  2 => 0.285714
  3 => 0.333333
  1 => 0.0

In [10]:
coef, rhs = coefficient_GMI_original_variable(
    coefficient_set, 
    cols,
    A,
    b
)

(Dict(2 => -0.9523809523809527, 1 => -2.3809523809523827), -5.238095238095242)

In [11]:
@constraint(model, sum(coef[k] * x[k] for k in 1:cols) ≥ rhs)

-2.3809523809523827 x[1] - 0.9523809523809527 x[2] ≥ -5.238095238095242

In [12]:
model = direct_model(Gurobi.Optimizer())

# 定义变量：x 为连续变量，y 为整数变量
@variable(model, x >= 0)
@variable(model, y >= 0)
@variable(model, s ≥ 0)

# 添加约束
@constraint(model, x + y + s == 10.3)        # 等式约束
@constraint(model, 2x - y <= 5)        # 小于等于约束
@constraint(model, x - 3y >= -4)       # 大于等于约束

# 目标函数（例如最小化 x + y）
@objective(model, Min, x + y + 10s)

# 求解模型
optimize!(model)
grb = backend(model)
b_ind = Cint[0, 1, 2]                 # indices of nonzeros
b_val = Cdouble[10.3, 5, -4]              # corresponding values

Gurobi Optimizer version 10.0.1 build v10.0.1rc0 (mac64[arm])

CPU model: Apple M3 Max
Thread count: 16 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 3 rows, 3 columns and 7 nonzeros
Model fingerprint: 0x967fab18
Coefficient statistics:
  Matrix range     [1e+00, 3e+00]
  Objective range  [1e+00, 1e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [4e+00, 1e+01]
Presolve time: 0.00s
Presolved: 3 rows, 3 columns, 7 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    1.0300000e+01   7.800000e+00   0.000000e+00      0s
       2    4.5400000e+01   0.000000e+00   0.000000e+00      0s

Solved in 2 iterations and 0.00 seconds (0.00 work units)
Optimal objective  4.540000000e+01

User-callback calls 42, time in user-callback 0.00 sec


3-element Vector{Float64}:
 10.3
  5.0
 -4.0

In [13]:
x_ind, x_val = my_GRBFSolve(grb, b_ind, b_val)
x_val = Dict(zip(x_ind, x_val))

ind, val = my_GRBBinvRowi(grb, 0)
val = Dict(zip(ind, val))

bhead, cols = my_getBasisHead(grb)

(Int32[2, 1, 0], 3)

In [26]:
value(s)

3.900000000000001

In [15]:
x_val

Dict{Int32, Float64} with 3 entries:
  0 => 3.9
  2 => 3.8
  1 => 2.6

In [16]:
ind, val = my_GRBBinvRowi(grb, 0)

(Int32[2, 3, 4, 5], [1.0, 1.0, -0.7999999999999999, 0.5999999999999999])

In [17]:
ind, val = my_GRBBinvRowi(grb, 1)

(Int32[1, 4, 5], [1.0, 0.19999999999999998, -0.39999999999999997])

In [18]:
ind, val = my_GRBBinvRowi(grb, 2)

(Int32[0, 4, 5], [1.0, 0.6, -0.19999999999999998])

In [19]:
# let i be the one with fractional x_val which in an integer
i = 0
basic_index = bhead[i+1]
b_i = x_val[i]

3.900000000000001

In [20]:
coefficient_set = coefficient_GMI_extended_variable( 
    b_i,
    val,
    [1,2]##TODO: integer set
)

BoundsError: BoundsError: attempt to access Float64 at index [2]